# Mirage Shrestha Week 6 Assignment

In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import pandas as pd
import numpy as np
import random

In [5]:
df = pd.read_csv('dataset/clean_french_data.csv')
 
# Converting pandas columns into standard Python lists
english_sentences = df['English'].tolist()
french_sentences  = df['French'].tolist()

In [11]:
SOS_token = "<SOS>"   # Start of Sequence
EOS_token = "<EOS>"  # End of Sequence

french_sentences  = [f"{SOS_token} {sentence} {EOS_token}" for sentence in french_sentences]
english_sentences = [f"{sentence} {EOS_token}"             for sentence in english_sentences]
 
print("Example English:", english_sentences[0])
print("Example French :", french_sentences[0])

Example English: go <EOS>
Example French : <SOS> va <EOS>


In [12]:
class Vocabulary:
    def __init__(self, name):
        self.name = name
        # SOS = 0, EOS = 1 are reserved from the start
        self.word2index = {"<SOS>": 0, "<EOS>": 1}
        self.index2word = {0: "<SOS>", 1: "<EOS>"}
        self.n_words = 2  # count starts at 2 because SOS and EOS are already added
 
    def add_sentence(self, sentence):
        """Break a sentence into words and add each one."""
        for word in sentence.split(' '):
            self.add_word(word)
 
    def add_word(self, word):
        """Add a single word to the vocabulary if it is not already present."""
        if word not in self.word2index:
            self.word2index[word] = self.n_words
            self.index2word[self.n_words] = word
            self.n_words += 1

In [13]:
english_vocab = Vocabulary("English")
french_vocab  = Vocabulary("French")
 
for sentence in english_sentences:
    english_vocab.add_sentence(sentence)
 
for sentence in french_sentences:
    french_vocab.add_sentence(sentence)
 
print(f"\nTotal unique English words : {english_vocab.n_words}")
print(f"Total unique French words  : {french_vocab.n_words}")
 
# Quick sanity check
test_word = "bonjour"
if test_word in french_vocab.word2index:
    wid = french_vocab.word2index[test_word]
    print(f"\nThe ID for '{test_word}' is: {wid}")
    print(f"Reversing ID {wid} gives us: '{french_vocab.index2word[wid]}'")


Total unique English words : 283
Total unique French words  : 385


In [14]:
def indexes_from_sentence(vocab, sentence):
    """Convert a sentence string into a list of word IDs."""
    return [vocab.word2index[word] for word in sentence.split(' ')]
 
def tensor_from_sentence(vocab, sentence):
    """Convert a sentence string into a PyTorch tensor of word IDs."""
    indexes = indexes_from_sentence(vocab, sentence)
    return torch.tensor(indexes, dtype=torch.long).view(-1, 1)
 
def tensors_from_pair(english_sentence, french_sentence):
    """Convert an English/French pair into (input_tensor, target_tensor)."""
    input_tensor  = tensor_from_sentence(english_vocab, english_sentence)
    target_tensor = tensor_from_sentence(french_vocab,  french_sentence)
    return (input_tensor, target_tensor)
 
# Testing on the very first sentence pair
sample_english = english_sentences[0]
sample_french  = french_sentences[0]
 
print(f"\nOriginal English : '{sample_english}'")
print(f"Original French  : '{sample_french}'")
 
input_tensor, target_tensor = tensors_from_pair(sample_english, sample_french)
 
print("\n--- Converted English Tensor ---")
print(input_tensor)
print(f"Shape: {input_tensor.shape}")
 
print("\n--- Converted French Tensor ---")
print(target_tensor)
print(f"Shape: {target_tensor.shape}")


Original English : 'go <EOS>'
Original French  : '<SOS> va <EOS>'

--- Converted English Tensor ---
tensor([[2],
        [1]])
Shape: torch.Size([2, 1])

--- Converted French Tensor ---
tensor([[0],
        [2],
        [1]])
Shape: torch.Size([3, 1])


In [15]:
class EncoderRNN(nn.Module):
    def __init__(self, input_size, hidden_size):
        super(EncoderRNN, self).__init__()
        self.hidden_size = hidden_size
 
        # 1. Embedding Layer
        #    input_size  = total words in the English vocabulary
        #    hidden_size = dimension of the dense word vector
        self.embedding = nn.Embedding(input_size, hidden_size)
 
        # 2. Recurrent Layer (GRU)
        #    GRU is a lighter alternative to LSTM with comparable performance
        self.gru = nn.GRU(hidden_size, hidden_size)
 
    def forward(self, input, hidden):
        # Embed the current word ID → dense vector, reshape for GRU: (1, 1, hidden)
        embedded = self.embedding(input).view(1, 1, -1)
 
        # Feed embedded word + previous hidden state through GRU
        output, hidden = self.gru(embedded, hidden)
 
        return output, hidden
 
    def initHidden(self):
        """Zero out the hidden state before reading a new sentence."""
        return torch.zeros(1, 1, self.hidden_size)
 
# Initializing the Encoder (hidden_size = 256 keeps it light yet expressive)
hidden_size = 256
encoder = EncoderRNN(input_size=english_vocab.n_words, hidden_size=hidden_size)
 
print("\nEncoder Architecture:")
print(encoder)
 



Encoder Architecture:
EncoderRNN(
  (embedding): Embedding(283, 256)
  (gru): GRU(256, 256)
)


In [16]:

class DecoderRNN(nn.Module):
    def __init__(self, hidden_size, output_size):
        super(DecoderRNN, self).__init__()
        self.hidden_size = hidden_size
 
        # 1. Embedding Layer
        #    output_size = total words in the French vocabulary
        self.embedding = nn.Embedding(output_size, hidden_size)
 
        # 2. Recurrent Layer (GRU)
        self.gru = nn.GRU(hidden_size, hidden_size)
 
        # 3. Output Layer — maps 256-dim hidden state → full French vocabulary scores
        self.out = nn.Linear(hidden_size, output_size)
 
    def forward(self, input, hidden):
        # 1. Embed the current word
        output = self.embedding(input).view(1, 1, -1)
 
        # ReLU helps the network learn more complex non-linear patterns
        output = F.relu(output)
 
        # 2. Pass through GRU with the previous hidden state
        output, hidden = self.gru(output, hidden)
 
        # 3. Project to vocabulary size and convert to log-probabilities
        output = self.out(output[0])
        output = F.log_softmax(output, dim=1)
 
        return output, hidden
 
# Initializing the Decoder (hidden_size must match Encoder for seamless handoff)
decoder = DecoderRNN(hidden_size=hidden_size, output_size=french_vocab.n_words)
 
print("\nDecoder Architecture:")
print(decoder)


Decoder Architecture:
DecoderRNN(
  (embedding): Embedding(385, 256)
  (gru): GRU(256, 256)
  (out): Linear(in_features=256, out_features=385, bias=True)
)


In [17]:
criterion = nn.NLLLoss()
 
# Teacher Forcing: 50 % of the time feed the correct word; rest use model's own guess
teacher_forcing_ratio = 0.5
 
 
def train_step(input_tensor, target_tensor,
               encoder, decoder,
               encoder_optimizer, decoder_optimizer,
               criterion):
    """Run one training step on a single sentence pair."""
 
    # 1. Clear accumulated gradients from the previous step
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
 
    input_length  = input_tensor.size(0)
    target_length = target_tensor.size(0)
 
    # 2. Initialize Encoder hidden state to zeros
    encoder_hidden = encoder.initHidden()
    loss = 0
 
    # 3. ENCODER PHASE — read the English sentence word by word
    for ei in range(input_length):
        encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
 
    # 4. PASSING THE BATON
    #    The Encoder's final hidden state becomes the Decoder's context vector
    decoder_hidden = encoder_hidden
 
    # The Decoder always starts with the <SOS> token (ID = 0)
    decoder_input = torch.tensor([[0]], dtype=torch.long)
 
    # 5. DECODER PHASE — decide whether to use Teacher Forcing
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False
 
    if use_teacher_forcing:
        # Feed the CORRECT target word as next input at every step
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            loss += criterion(decoder_output, target_tensor[di])
            decoder_input = target_tensor[di]   # next input = ground-truth word
 
    else:
        # Feed the model's OWN GUESS as next input
        for di in range(target_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.topk(1)
            decoder_input = topi.squeeze().detach()  # detach so gradients don't flow back
 
            loss += criterion(decoder_output, target_tensor[di])
 
            # Stop early if the model predicts <EOS> (ID = 1)
            if decoder_input.item() == 1:
                break
 
    # 6. Backpropagation — compute gradients and update weights
    loss.backward()
    encoder_optimizer.step()
    decoder_optimizer.step()
 
    # Return average loss per target word
    return loss.item() / target_length
 
 
def train_iters(encoder, decoder, n_iters, print_every=1000, learning_rate=0.01):
    """Train the Encoder-Decoder model for n_iters iterations."""
 
    # 1. Stochastic Gradient Descent optimizers for both networks
    encoder_optimizer = optim.SGD(encoder.parameters(), lr=learning_rate)
    decoder_optimizer = optim.SGD(decoder.parameters(), lr=learning_rate)
 
    print_loss_total = 0
 
    # 2. Randomly sample n_iters sentence pairs from the dataset
    print(f"Preparing {n_iters} training pairs...")
    training_pairs = []
    for _ in range(n_iters):
        idx  = random.randint(0, len(english_sentences) - 1)
        pair = tensors_from_pair(english_sentences[idx], french_sentences[idx])
        training_pairs.append(pair)
 
    print("Starting training...\n")
 
    # 3. Main training loop
    for iter in range(1, n_iters + 1):
        training_pair = training_pairs[iter - 1]
        input_tensor  = training_pair[0]
        target_tensor = training_pair[1]
 
        loss = train_step(input_tensor, target_tensor,
                          encoder, decoder,
                          encoder_optimizer, decoder_optimizer,
                          criterion)
 
        print_loss_total += loss
 
        if iter % print_every == 0:
            print_loss_avg    = print_loss_total / print_every
            print_loss_total  = 0
            print(f"Iteration: {iter}/{n_iters} | Average Loss: {print_loss_avg:.4f}")
 
# Train for 10,000 iterations
train_iters(encoder, decoder, n_iters=10000, print_every=1000)

Preparing 10000 training pairs...
Starting training...

Iteration: 1000/10000 | Average Loss: 2.8410
Iteration: 2000/10000 | Average Loss: 2.1465
Iteration: 3000/10000 | Average Loss: 1.6089
Iteration: 4000/10000 | Average Loss: 1.1291
Iteration: 5000/10000 | Average Loss: 0.7840
Iteration: 6000/10000 | Average Loss: 0.5818
Iteration: 7000/10000 | Average Loss: 0.4058
Iteration: 8000/10000 | Average Loss: 0.2952
Iteration: 9000/10000 | Average Loss: 0.1915
Iteration: 10000/10000 | Average Loss: 0.1368


In [28]:
def evaluate(encoder, decoder, sentence, max_length=15):
    """
    Translate a single English sentence to French.
    The sentence must already have <EOS> appended.
    """
    with torch.no_grad():
        # Prepare input tensor
        input_tensor  = tensor_from_sentence(english_vocab, sentence)
        input_length  = input_tensor.size()[0]
 
        # Run through Encoder
        encoder_hidden = encoder.initHidden()
        for ei in range(input_length):
            encoder_output, encoder_hidden = encoder(input_tensor[ei], encoder_hidden)
 
        # Hand context vector to Decoder; start with <SOS>
        decoder_input  = torch.tensor([[0]], dtype=torch.long)
        decoder_hidden = encoder_hidden
 
        decoded_words = []
 
        # Decoder generates one word at a time
        for di in range(max_length):
            decoder_output, decoder_hidden = decoder(decoder_input, decoder_hidden)
            topv, topi = decoder_output.data.topk(1)
 
            if topi.item() == 1:                          # <EOS> predicted → stop
                decoded_words.append('<EOS>')
                break
            else:
                decoded_words.append(french_vocab.index2word[topi.item()])
 
            decoder_input = topi.squeeze().detach()
 
        return ' '.join(decoded_words)
 
 
def evaluateRandomly(encoder, decoder, n=5):
    """Evaluate the model on n randomly selected sentence pairs."""
    for i in range(n):
        idx          = random.randint(0, len(english_sentences) - 1)
        eng_sentence = english_sentences[idx]
        fra_sentence = french_sentences[idx]
 
        print('> (English input)    :', eng_sentence)
        print('= (Expected French)  :', fra_sentence)
 
        output_words = evaluate(encoder, decoder, eng_sentence)
        print('< (Model translation):', output_words)
        print()
 
 
print("\n" + "="*60)
print("Evaluating trained model on random sentence pairs...")
print("="*60 + "\n")
evaluateRandomly(encoder, decoder)


Evaluating trained model on random sentence pairs...

> (English input)    : where are you <EOS>
= (Expected French)  : <SOS> où êtes-vous <EOS>
< (Model translation): <SOS> où êtes-vous <EOS>

> (English input)    : mary is tired <EOS>
= (Expected French)  : <SOS> marie est fatiguée <EOS>
< (Model translation): <SOS> marie est fatiguée <EOS>

> (English input)    : she waited <EOS>
= (Expected French)  : <SOS> elle a attendu <EOS>
< (Model translation): <SOS> elle a attendu <EOS>

> (English input)    : he is quiet <EOS>
= (Expected French)  : <SOS> il est calme <EOS>
< (Model translation): <SOS> il est calme <EOS>

> (English input)    : leave <EOS>
= (Expected French)  : <SOS> pars <EOS>
< (Model translation): <SOS> pars <EOS>

